# Glassbox bounded headline reproduction

This notebook runs the held-out Qwen2.5-1.5B bounded audit behind the repository headline: residual localization, a trained SAE, matched controls, causal interventions, and an audit card. On a Colab T4, the intended budget is about 40 minutes and well below $10, but runtime and Colab pricing are not guarantees. The result remains `empirical_bounded_real_audit`; it does not replace the committed 1,000-pair preregistered evidence.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path('/content/glassbox-audit').exists():
    subprocess.run(['git', 'clone', 'https://github.com/HassanHassnain/glassbox-audit.git', '/content/glassbox-audit'], check=True)
os.chdir('/content/glassbox-audit')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[paper,accelerate,backends,sae-zoo]'], check=True)
subprocess.run(['glassbox', 'build-real-audit-data', '--output', 'data/refusal_controlled_v1.jsonl'], check=True)

In [ ]:
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print(torch.cuda.get_device_name(0))
print(f'free={free_bytes / 2**30:.1f} GiB total={total_bytes / 2**30:.1f} GiB')
assert free_bytes >= 10 * 2**30, 'At least 10 GiB free GPU memory is required.'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

The default uses raw Hugging Face hooks because the headline component and head analyses require them. To validate an ecosystem backend on the same residual protocol, copy the config and change `model.backend` to `nnsight` or `transformer_lens`.

In [ ]:
import time

config = 'configs/qwen2.5-1.5b-real-audit.yaml'
output = 'artifacts/colab-headline'
started = time.time()
subprocess.run(['glassbox', 'run', '--config', config, '--output', output], check=True)
print(f'audit minutes: {(time.time() - started) / 60:.1f}')

In [ ]:
import json

import pandas as pd

summary = json.loads(Path(output, 'summary.json').read_text())
manifest = json.loads(Path(output, 'manifest.json').read_text())
tests = json.loads(Path(output, 'interventions.json').read_text())['causal_tests']
assert manifest['config']['evidence_class'] == 'empirical_bounded_real_audit'
rows = []
for method, result in tests.items():
    rows.append({
        'method': method,
        'harmful score delta': result['delta_mean_harmful_score_ci']['estimate'],
        'benign score delta': result['delta_mean_benign_score_ci']['estimate'],
        'capability NLL delta': result['delta_capability_nll_ci']['estimate'],
    })
print('target layer:', summary['target_layer'])
pd.DataFrame(rows).sort_values('harmful score delta')

In [ ]:
subprocess.run(['glassbox', 'paper', '--artifacts', output, '--docs', f'{output}/paper'], check=True)
subprocess.run([
    'glassbox', 'certify', '--artifacts', output,
    '--output-json', f'{output}/audit-card.json',
    '--output-html', f'{output}/audit-card.html',
], check=True)
print(Path(output, 'REPORT.md').read_text()[:4000])

Preserve the full artifact directory, including failed controls and the manifest. Compare the method ordering and confidence intervals with the committed bounded artifact; do not promote this run to the preregistered 1,000-pair evidence class.